# Equivalent Circuit Model (ECM) for Battery Analysis

This notebook demonstrates ECM modeling for battery State of Health (SoH) estimation.

## What is ECM?

ECM (Equivalent Circuit Model) represents battery behavior using electrical circuit elements:
- **Resistors (R)**: Internal resistance, polarization resistance
- **Capacitors (C)**: Charge storage, polarization effects
- **Voltage Source (OCV)**: Open Circuit Voltage

## Models Implemented:

1. **Rint Model**: Simplest model with only internal resistance
2. **RC Model (Thevenin)**: First-order RC circuit
3. **2RC Model (PNGV)**: Second-order RC circuit with fast and slow dynamics

In [ ]:
# Import libraries
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from battery_loader import load_data
from ecm_model import RintModel, RCModel, TwoRCModel, SoCEstimator, calculate_model_metrics, plot_ecm_results
from ecm_parameter_extraction import ECMParameterExtractor, compare_ecm_models

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Load Battery Data

In [ ]:
# Load battery data
battery_id = 'B0005'
dataset, capacity_data = load_data(battery_id)

print(f"Battery: {battery_id}")
print(f"Total discharge cycles: {len(capacity_data)}")
print(f"Total measurements: {len(dataset)}")
print(f"\nDataset columns: {list(dataset.columns)}")

## 2. Visualize Battery Degradation

In [ ]:
# Plot capacity degradation
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(capacity_data['cycle'], capacity_data['capacity'], 'b-o', markersize=3)
plt.xlabel('Cycle Number', fontsize=12)
plt.ylabel('Capacity (Ah)', fontsize=12)
plt.title(f'Battery {battery_id} - Capacity Degradation', fontsize=14)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
initial_capacity = capacity_data['capacity'].iloc[0]
soh = capacity_data['capacity'] / initial_capacity
plt.plot(capacity_data['cycle'], soh * 100, 'r-o', markersize=3)
plt.xlabel('Cycle Number', fontsize=12)
plt.ylabel('State of Health (%)', fontsize=12)
plt.title(f'Battery {battery_id} - SoH Degradation', fontsize=14)
plt.axhline(y=80, color='k', linestyle='--', label='80% Threshold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Compare ECM Models on a Single Cycle

In [ ]:
# Compare all three ECM models
cycle_to_analyze = 50
extractor, rint_model, rc_model, tworc_model = compare_ecm_models(battery_id, cycle_to_analyze)

## 4. Visualize Model Predictions

In [ ]:
# Extract cycle data for visualization
cycle_data = extractor.extract_cycle_data(cycle_to_analyze)
soc = extractor.calculate_soc(cycle_data)

# Get predictions from each model
v_rint = rint_model.predict(cycle_data['current'], soc)
v_rc = rc_model.predict(cycle_data['current'], soc, cycle_data['time'])
v_2rc = tworc_model.predict(cycle_data['current'], soc, cycle_data['time'])

# Plot comparison
plt.figure(figsize=(14, 10))

# Voltage comparison
plt.subplot(4, 1, 1)
plt.plot(cycle_data['time'], cycle_data['voltage'], 'k-', label='Measured', linewidth=2)
plt.plot(cycle_data['time'], v_rint, 'b--', label='Rint Model', linewidth=1.5)
plt.plot(cycle_data['time'], v_rc, 'g--', label='RC Model', linewidth=1.5)
plt.plot(cycle_data['time'], v_2rc, 'r--', label='2RC Model', linewidth=1.5)
plt.ylabel('Voltage (V)', fontsize=12)
plt.title(f'ECM Model Comparison - Cycle {cycle_to_analyze}', fontsize=14)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)

# Current
plt.subplot(4, 1, 2)
plt.plot(cycle_data['time'], cycle_data['current'], 'b-', linewidth=1.5)
plt.ylabel('Current (A)', fontsize=12)
plt.title('Current Profile', fontsize=14)
plt.grid(True, alpha=0.3)

# SoC
plt.subplot(4, 1, 3)
plt.plot(cycle_data['time'], soc * 100, 'g-', linewidth=1.5)
plt.ylabel('SoC (%)', fontsize=12)
plt.title('State of Charge', fontsize=14)
plt.grid(True, alpha=0.3)

# Errors
plt.subplot(4, 1, 4)
error_rint = (cycle_data['voltage'] - v_rint) * 1000
error_rc = (cycle_data['voltage'] - v_rc) * 1000
error_2rc = (cycle_data['voltage'] - v_2rc) * 1000
plt.plot(cycle_data['time'], error_rint, 'b-', label='Rint Error', linewidth=1.5)
plt.plot(cycle_data['time'], error_rc, 'g-', label='RC Error', linewidth=1.5)
plt.plot(cycle_data['time'], error_2rc, 'r-', label='2RC Error', linewidth=1.5)
plt.xlabel('Time (s)', fontsize=12)
plt.ylabel('Error (mV)', fontsize=12)
plt.title('Prediction Errors', fontsize=14)
plt.legend(loc='best')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Extract ECM Parameters Over Battery Lifetime

In [ ]:
# Extract RC model parameters over lifetime
# This will take a few minutes...
extractor = ECMParameterExtractor(battery_id)
extractor.load_battery_data()

# Extract parameters every 10 cycles
params_df = extractor.extract_parameters_over_lifetime(model_type='RC', cycle_step=10)

print("\nParameter extraction complete!")
print(f"\nExtracted parameters for {len(params_df)} cycles")
print("\nFirst few rows:")
print(params_df.head())

## 6. Visualize Parameter Degradation

In [ ]:
# Plot parameter degradation
fig = extractor.plot_parameter_degradation(save_path='pics/ecm_parameter_degradation.png')
plt.show()

## 7. Analyze Parameter Trends

In [ ]:
# Statistical analysis of parameter changes
print("Parameter Statistics:")
print("=" * 60)

for param in ['R0', 'R1', 'C1', 'tau1']:
    if param in params_df.columns:
        initial = params_df[param].iloc[0]
        final = params_df[param].iloc[-1]
        change = ((final - initial) / initial) * 100
        
        print(f"\n{param}:")
        print(f"  Initial: {initial:.6f}")
        print(f"  Final: {final:.6f}")
        print(f"  Change: {change:+.2f}%")

# Capacity change
initial_cap = params_df['capacity'].iloc[0]
final_cap = params_df['capacity'].iloc[-1]
cap_change = ((final_cap - initial_cap) / initial_cap) * 100

print(f"\nCapacity:")
print(f"  Initial: {initial_cap:.4f} Ah")
print(f"  Final: {final_cap:.4f} Ah")
print(f"  Change: {cap_change:+.2f}%")

## 8. Correlation Between Parameters and Capacity

In [ ]:
# Calculate correlations
import seaborn as sns

# Select numeric columns
numeric_cols = ['capacity', 'R0', 'R1', 'C1', 'tau1', 'RMSE']
corr_data = params_df[numeric_cols]

# Correlation matrix
plt.figure(figsize=(10, 8))
correlation_matrix = corr_data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Between ECM Parameters and Capacity', fontsize=14)
plt.tight_layout()
plt.show()

print("\nCorrelation with Capacity:")
print("=" * 40)
capacity_corr = correlation_matrix['capacity'].sort_values(ascending=False)
for param, corr in capacity_corr.items():
    if param != 'capacity':
        print(f"{param:<10}: {corr:+.4f}")

## 9. Save Results

In [ ]:
# Save parameters to CSV
extractor.save_parameters(f'{battery_id}_ecm_parameters.csv')

print("\n✅ ECM analysis complete!")
print(f"\nResults saved:")
print(f"  - Parameters: {battery_id}_ecm_parameters.csv")
print(f"  - Plots: pics/ecm_parameter_degradation.png")

## 10. Key Insights

### ECM Parameter Interpretation:

1. **R0 (Ohmic Resistance)**:
   - Represents immediate voltage drop
   - Increases with aging (SEI layer growth, electrolyte degradation)
   - Strong indicator of battery health

2. **R1 (Polarization Resistance)**:
   - Represents charge transfer resistance
   - Increases with aging (active material loss)
   - Related to power capability

3. **C1 (Polarization Capacitance)**:
   - Represents charge storage in double layer
   - Decreases with aging (surface area reduction)
   - Related to transient response

4. **τ1 = R1 × C1 (Time Constant)**:
   - Represents response time
   - Changes indicate degradation mechanisms

### Applications:

- **SoH Estimation**: Use R0 increase as health indicator
- **RUL Prediction**: Model parameter degradation trends
- **Fault Detection**: Abnormal parameter changes indicate faults
- **Real-time Monitoring**: Fast computation for online estimation